# 04 — Reflection & Verification

*Level 5 — Agentic RAG*

## Objective
Check an answer two different ways: **source checking** (is every claim traceable to the retrieved evidence?) and **ground-truth verification** (does the answer actually match the real accepted answer?) — the first is always available, the second only during evaluation, since TriviaQA's answer aliases are what make it possible at all.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "tools", "verification", "reflection"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from agentic_common.dataset import prepare
from agentic_common.retrieval import DenseRetriever
from agentic_common.llm import OllamaLLM
from vector_tool import VectorTool, GetDocumentTool
from agents.rag_agent import RAGAgent
from answer_verifier import verify_answer

data = prepare()
corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)
tools = {"vector_search": VectorTool(retriever, data.corpus), "get_document": GetDocumentTool(data)}


## Run the agent on several real questions and verify against real ground truth


In [3]:
agent = RAGAgent(tools=tools, max_steps=3)
sample_qids = list(data.questions.keys())[:8]

n_correct = 0
for qid in sample_qids:
    q = data.questions[qid]
    state = agent.run(q["question"])
    correct = verify_answer(state.answer, q["aliases"] + [q["answer"].lower()])
    n_correct += correct
    print(f"{'CORRECT' if correct else 'WRONG  '} | sources_verified={state.verified!s:<5} | {q['question'][:55]}")

print(f"\n{n_correct}/{len(sample_qids)} answers matched real ground truth.")


CORRECT | sources_verified=True  | A sophomore is a student in which year of a US college?


CORRECT | sources_verified=True  | Sara Jane Moore was responsible for an unsuccessful att


CORRECT | sources_verified=True  | What is the name of the assembly of cardinals for the e


WRONG   | sources_verified=False | Which cocktail consists of rum, curacao and lime juice?


CORRECT | sources_verified=False | What was the occupation of Mr. Sowerberry, in ‘Oliver T


WRONG   | sources_verified=True  | Melanie Molitor is the mom of which tennis world NO 1?


CORRECT | sources_verified=False | Which builder of steam engines formed a successful part


CORRECT | sources_verified=True  | Who also writes using the pseudonym 'Barbara Vine'?

6/8 answers matched real ground truth.


## What I observed

**6 of 8 real answers (75%) matched TriviaQA's actual ground truth** — this is the first level in this repository where correctness is checked against the *real answer text*, not just "did we retrieve the right document."

The two verification signals disagreed in both directions on this run, which is exactly why both exist:

- **Correct answer, `sources_verified=False`** (the Oliver Twist and steam-engines questions) — the answer was actually right, but the source-checking LLM call didn't confirm it that round. A correct answer was almost flagged as unsupported.
- **Wrong answer, `sources_verified=True`** (the tennis question, "Melanie Molitor is the mom of which tennis world No. 1?") — the source checker judged the answer well-supported by the retrieved evidence, and it still didn't match the real answer. Likely cause: the retrieved evidence itself didn't contain the correct fact, so the answer was *internally consistent with bad evidence* — support-checking cannot catch that; only ground-truth verification can.

Neither signal alone is trustworthy. Source checking answers "is this answer make-believe relative to what was retrieved?"; ground-truth verification answers "is this actually right?" — and only the second one is available outside of evaluation with a labeled dataset like this one.

## Next

[Level 6 — Multi-Agent RAG](../../06-multi-agent-rag/README.md) — one agent with several tools becomes several specialized agents coordinated by a supervisor.
